# Chapter 17. FIFA 월드컵 경기 데이터로 배우는 데이터 분석 실습

이 노트북은 실제 CSV 파일인 `fifa_world_cup_all_matches_1930_2026.csv`를 활용해 데이터 분석을 실습하는 자료입니다.

이번 장은 고등학생이 데이터 분석 프로젝트를 이해하는 데 초점을 맞춥니다. 실제 데이터는 단순한 숫자 모음이 아니라 다음 질문을 해결하는 도구입니다.

- 어느 팀이 가장 많이 이겼는가?
- 월드컵 대회별 득점 경향은 어떠한가?
- 홈 팀이 유리한가?
- 특정 대회에서는 어떤 팀이 강했는가?
- 경기 수와 득점 수 사이에는 어떤 관계가 있는가?

이런 질문을 만들고, 데이터를 정리하고, 그래프와 통계를 통해 의미를 찾아보는 것이 바로 데이터 분석입니다.

## 1. 분석 흐름: 문제 정의 → 데이터 수집 → 전처리 → EDA → 해석과 보고

데이터 분석은 한 번에 끝나지 않습니다. 아래 흐름을 따라야 합니다.

1. 문제 정의
2. 데이터 수집
3. 전처리
4. 탐색적 분석(EDA)
5. 해석과 보고

이번 실습에서는 월드컵 경기 데이터 파일을 이용해 이 흐름을 직접 따라가 봅니다.

In [ ]:
import pandas as pd

analysis_steps = [
    "Problem Definition",
    "Data Collection",
    "Data Cleaning",
    "Exploratory Data Analysis",
    "Interpretation and Report",
]

for i, step in enumerate(analysis_steps, start=1):
    print(f"{i}. {step}")

## 2. 문제 정의: 무엇을 알아보고 싶은가?

데이터를 보기 전에 질문을 정해야 합니다. 이 단계가 중요합니다. 질문이 없으면 데이터는 단순한 표에 지나지 않습니다.

월드컵 데이터를 분석할 때 아래와 같은 질문을 만들 수 있습니다.

- 어느 나라가 월드컵에서 가장 많이 우승했는가?
- 특정 나라가 홈에서 더 좋은 성적을 냈는가?
- 대회별 평균 득점은 어떻게 변화했는가?
- 팀1과 팀2의 득점 분포는 어떤가?
- 특정 대회에서 경기 수와 총 득점 수는 어떤 관계가 있는가?

실제 데이터 분석에서는 질문을 정한 뒤에 데이터를 탐색합니다.

In [ ]:
questions = {
    "Q1": "Which country has won the most World Cup titles?",
    "Q2": "Do host countries perform better at home?",
    "Q3": "How has the average goals per match changed over time?",
    "Q4": "Which teams have the highest total goals?",
    "Q5": "Do matches with more goals tend to be in certain stages?",
}

for key, value in questions.items():
    print(f"{key}: {value}")

## 3. 데이터 수집: CSV 파일 불러오기

이 실습에서는 `fifa_world_cup_all_matches_1930_2026.csv` 파일을 사용합니다. 이 파일에는 월드컵 경기마다 다음 정보가 포함됩니다.

- 대회 연도
- 개최국
- 경기 날짜
- 팀1, 팀2
- 전반전 점수
- 후반전 점수
- 승자
- 경기장, 도시
- 총 득점

CSV 파일은 데이터 분석에서 가장 많이 쓰는 형식입니다.

In [ ]:
# CSV 불러오기
file_path = "fifa_world_cup_all_matches_1930_2026.csv"
df = pd.read_csv(file_path)

print(df.head())
print("\nRows:", len(df))
print("Columns:", list(df.columns))

## 4. 전처리: 분석에 적합하게 정리하기

데이터를 불러오면 바로 분석하진 않습니다. 먼저 다음을 살펴봐야 합니다.

- 결측값이 있는지 확인
- 날짜 형식이 올바른지 확인
- 숫자 컬럼이 숫자로 인식되는지 확인
- 불필요한 열을 정리
- 새로 계산할 변수를 만들기

이번 데이터는 경기 결과와 득점 수가 정리되어 있으므로, 몇 가지 유용한 변수를 만들어 분석을 더 쉽게 만들 수 있습니다.

In [ ]:
# 결측값 확인
print(df.isna().sum().head(20))

# 총 득점 수 계산
# 기존에도 total_goals_team1, total_goals_team2가 있으므로 이를 활용해 총 경기 득점 계산
# 여기서는 팀1과 팀2 모두 포함한 합계를 새 컬럼으로 만듭니다.
df["total_goals_match"] = df["total_goals_team1"] + df["total_goals_team2"]

# 경기 결과를 간단하게 저장하는 새 컬럼 만들기
# 해당 경기의 결과가 1이면 팀1 승, 2이면 팀2 승, 0이면 무승부
# 이 값은 예시이며 실제 분석 목적에 맞게 확장할 수 있습니다.
df["result_code"] = df["winner"].map({
    df["team1"].iloc[0]: 1,
}) if len(df) > 0 else 0

# 결과 코드 컬럼을 계산하기 위해 안전한 형태로 다시 작성
result_map = {}
for team1, team2, winner in zip(df["team1"], df["team2"], df["winner"]):
    if winner == team1:
        result_map[(team1, team2)] = 1
    elif winner == team2:
        result_map[(team1, team2)] = 2
    else:
        result_map[(team1, team2)] = 0

# 실제로는 사용하지 않을 수 있지만, 보조 컬럼을 만드는 예시입니다.
df["match_result"] = [result_map.get((team1, team2), 0) for team1, team2 in zip(df["team1"], df["team2"])]

print(df[["team1", "team2", "winner", "total_goals_match", "match_result"]].head())

## 5. EDA: 데이터의 패턴 찾기

EDA는 그래프와 통계로 데이터를 살펴보는 단계입니다. 여기서 우리는 어떤 패턴이 보이는지 확인합니다.

예를 들어 다음을 살펴볼 수 있습니다.

- 연도별 평균 득점 수
- 가장 많은 골을 넣은 나라
- 어떤 팀이 가장 많은 경기를 이겼는가?
- 홈 팀이 강했는지 여부

In [ ]:
# 연도별 평균 득점 수 계산
yearly_goals = df.groupby("world_cup_year")["total_goals_match"].mean().reset_index()
print(yearly_goals.head())

# 가장 많은 경기 득점 기록을 보인 대회 확인
print("Highest average goals per match:")
print(yearly_goals.sort_values("total_goals_match", ascending=False).head())

In [ ]:
# 팀별 총 득점과 승리 수를 확인하는 예시
team_goals = pd.concat([
    df[["team1", "total_goals_team1"]].rename(columns={"team1": "team", "total_goals_team1": "goals"}),
    df[["team2", "total_goals_team2"]].rename(columns={"team2": "team", "total_goals_team2": "goals"}),
], ignore_index=True)

team_goals_summary = team_goals.groupby("team")["goals"].sum().sort_values(ascending=False)
print(team_goals_summary.head(10))

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "DejaVu Sans"

# 연도별 평균 득점 변화 그래프
plt.figure(figsize=(10, 5))
plt.plot(yearly_goals["world_cup_year"], yearly_goals["total_goals_match"], marker="o", color="steelblue")
plt.title("Average Goals per Match by World Cup Year")
plt.xlabel("World Cup Year")
plt.ylabel("Average Goals per Match")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## 6. EDA 해석: 의미를 찾기

그래프를 보고 의미를 해석하는 것이 핵심입니다. 예를 들어 다음과 같은 해석을 만들 수 있습니다.

- 대회별 평균 득점 수가 높을 때는 공격적이고 높은 점수 경기의 비율이 높다는 뜻일 수 있습니다.
- 어떤 팀은 거의 모든 경기에서 득점을 많이 넣어 강한 공격력을 보입니다.
- 특정 시기에는 경기 방식이 바뀌어 득점이 줄거나 줄어드는 흐름이 나타날 수 있습니다.

중요한 것은 "숫자만 보는 것"이 아니라 "그 숫자가 의미하는 바를 설명하는 것"입니다.

In [ ]:
# 간단한 해석 예시
print("Interpretation example:")
print("- A higher average goals value suggests matches were more attack-oriented.")
print("- Teams with large total goals often show strong offensive performance.")
print("- Comparing yearly trends helps us understand changes in playing style.")

## 7. 해석과 보고: 결과를 정리하기

데이터 분석의 마지막 단계는 결과를 정리해서 설명하는 것입니다.

좋은 보고서에는 다음 요소가 포함됩니다.

- 어떤 질문을 설정했는가?
- 어떤 데이터를 사용했는가?
- 어떤 전처리를 했는가?
- 어떤 패턴이 발견되었는가?
- 결과를 어떻게 해석하는가?
- 앞으로 어떤 질문을 더 탐색할 수 있는가?

예시 보고문:

> 월드컵 경기 데이터를 분석한 결과, 대회별 평균 득점 수는 연도에 따라 변동이 있었으며, 특정 시기에는 공격적인 경기가 많았음을 확인할 수 있었습니다. 팀별 전체 득점 합계 분석에서는 강한 공격력을 가진 팀들이 상위권에 위치했습니다. 이 결과는 경기 스타일 변화와 팀의 공격력 차이가 월드컵 결과에 영향을 미친다는 가설을 뒷받침합니다.

In [ ]:
report = {
    "Question": "What patterns exist in World Cup match data?",
    "Main Finding": "Average goals and team scoring totals vary by year and by team strength.",
    "Recommendation": "Use deeper analysis to compare tournament stages, host nations, and team-specific performance.",
}

for key, value in report.items():
    print(f"{key}: {value}")

## 8. 전체 실습 정리

이번 실습의 핵심은 월드컵 데이터로 데이터 분석을 실제로 수행해 본다는 점입니다.

단계별 핵심은 다음과 같습니다.

1. 문제 정의
2. 데이터 수집
3. 전처리
4. EDA
5. 해석과 보고

이 흐름은 학교 프로젝트, 사회 문제 분석, 스포츠 데이터 분석, 사업 데이터 분석까지 거의 모든 분야에 그대로 적용됩니다.

In [ ]:
process = [
    "Problem Definition",
    "Data Collection",
    "Data Cleaning",
    "Exploratory Data Analysis",
    "Interpretation and Report",
]

for index, step in enumerate(process, start=1):
    print(f"{index}. {step}")

assert len(process) == 5
print("Chapter 17 실습 검증 통과")

## 실습 과제

1. CSV 파일을 불러와 데이터 크기와 컬럼을 확인하세요.
2. `total_goals_match`를 만들고 평균값을 계산해 보세요.
3. 팀별 총 득점을 집계해 상위권 팀을 확인하세요.
4. 연도별 평균 득점 그래프를 그려 보세요.
5. 이 결과를 3~5문장으로 보고서 형태로 정리해 보세요.